In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
RUN_DOMAINS      = ''          # blank = the MD domain of NB_Config_MD_B1
ONLY             = ''          # blank = every table of those domains; or 'KNA1, MD.LFA1' (a table name, or DOMAIN.TABLE)
DRY_RUN_OVERRIDE = ''          # 'true' / 'false' for every domain and table; blank = the settings of the config
PIPELINE_RUN_ID  = ''          # a Fabric pipeline can pass one in
SWAP_SCOPE       = 'full'      # 'full': swap whole tables (rename); 'range': replace only the window of tables with 'range' settings

In [ ]:
%run ./NB_Config_MD_B1

In [ ]:
# ── Setup and the configuration, checked before anything runs ───────────────────
import os, uuid, time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
import pyspark.sql.functions as F
from delta.tables import DeltaTable

spark.conf.set('spark.sql.session.timeZone', 'UTC')       # all times in UTC, in Spark and in Python
os.environ['TZ'] = 'UTC'; time.tzset()

RUN_ID, RUN_AT = PIPELINE_RUN_ID or str(uuid.uuid4()), datetime.utcnow()
MAX_PARALLEL_DOMAINS = globals().get('MAX_PARALLEL_DOMAINS', 1)
LOG_TABLE_PATH       = globals().get('LOG_TABLE_PATH', '')
OVERRIDE = {'true': True, 'false': False}.get(DRY_RUN_OVERRIDE.strip().lower())      # None = use the config
SWAP_SCOPE = (globals().get('SWAP_SCOPE') or 'full').strip().lower()
RANGE_KEYS = {'year_col', 'period_col', 'periods'}

SETTING_KEYS = set(DEFAULTS)
DOMAIN_KEYS  = SETTING_KEYS | {'vl', 'lh_key', 'abfss', 'tables', 'enabled', 'log_abfss'}
TABLE_KEYS   = SETTING_KEYS - {'parallel_tables'}

def range_problems(where, rng):
    """A 'range' setting is a dict with year_col, period_col and periods (a whole number of periods, at least 1)."""
    if rng is None:
        return []
    if not isinstance(rng, dict) or set(rng) != RANGE_KEYS:
        return [f"{where}: 'range' needs exactly the keys {sorted(RANGE_KEYS)}"]
    if not isinstance(rng['periods'], int) or rng['periods'] < 1:
        return [f"{where}: 'range' periods must be a whole number, at least 1"]
    return []

def config_problems():
    """Every mistake in NB_Config_B1_Swap, all at once."""
    problems = []
    if SWAP_SCOPE not in ('full', 'range'):
        problems.append(f"SWAP_SCOPE: '{SWAP_SCOPE}' is not 'full' or 'range'")
    problems += range_problems('DEFAULTS', DEFAULTS.get('range'))
    for d, cfg in SWAP_DOMAINS.items():
        unknown = sorted(set(cfg) - DOMAIN_KEYS)
        if unknown:
            problems.append(f'{d}: unknown key(s) {unknown}')
        if not cfg.get('abfss') and not (cfg.get('vl') and cfg.get('lh_key')):
            problems.append(f"{d}: give 'abfss', or 'vl' and 'lh_key'")
        problems += range_problems(d, cfg.get('range'))
        tables = cfg.get('tables')
        if isinstance(tables, dict):
            for name, over in tables.items():
                bad = sorted(set(over or {}) - TABLE_KEYS)
                if bad:
                    problems.append(f'{d}.{name}: unknown or domain-only key(s) {bad}')
                problems += range_problems(f'{d}.{name}', (over or {}).get('range'))
        elif not isinstance(tables, list) or not tables:
            problems.append(f"{d}: 'tables' is a list, a dict, or ['*']")
        elif len(set(tables)) != len(tables):
            problems.append(f'{d}: a table is listed twice')
    for name in [d.strip() for d in RUN_DOMAINS.split(',') if d.strip()]:
        if name not in SWAP_DOMAINS:
            problems.append(f'RUN_DOMAINS: {name} is not in SWAP_DOMAINS')
    return problems

problems = config_problems()
if problems:
    raise ValueError('NB_Config_B1_Swap has problems:\n  ' + '\n  '.join(problems))

def settings(domain, table=None):
    """DEFAULTS, then the domain's keys, then the table's keys; DRY_RUN_OVERRIDE wins over all."""
    cfg = SWAP_DOMAINS[domain]
    s = dict(DEFAULTS)
    s.update({k: v for k, v in cfg.items() if k in SETTING_KEYS})
    if table and isinstance(cfg['tables'], dict):
        s.update(cfg['tables'].get(table) or {})
    if OVERRIDE is not None:
        s['dry_run'] = OVERRIDE
    return s

def range_start(periods):
    """Period key (YYYYPP) of the first period of the window: the run month and the periods - 1 before it.
    Assumes fiscal periods are calendar months (special periods 13-16 sort after period 12 of their year)."""
    m = RUN_AT.year * 12 + RUN_AT.month - 1 - (periods - 1)
    return (m // 12) * 100 + m % 12 + 1

domains = [d.strip() for d in RUN_DOMAINS.split(',') if d.strip()] or [d for d, c in SWAP_DOMAINS.items() if c.get('enabled', True)]
only    = [t.strip() for t in ONLY.split(',') if t.strip()]

print(f'Run {RUN_ID} · {RUN_AT:%Y-%m-%d %H:%M:%S} UTC · {len(domains)} domain(s), up to {MAX_PARALLEL_DOMAINS} at the same time · scope {SWAP_SCOPE}')
if SWAP_SCOPE == 'range':
    print('  range scope: only tables with range settings are swapped, and only their window; the others are skipped')
for d in domains:
    cfg, s = SWAP_DOMAINS[d], settings(d)
    tables = cfg['tables']
    shown = 'all staged tables' if tables == ['*'] else f'{len(tables)} table(s)'
    differs = ', '.join(f'{k}={s[k]}' for k in DEFAULTS if s[k] != DEFAULTS[k])
    print(f"  {d:<12} {'LIVE' if not s['dry_run'] else 'dry run':<8} {('address' if cfg.get('abfss') else cfg.get('vl')):<16} {shown:<18}" + (f'  {differs}' if differs else ''))
if only:
    print(f'Only: {only}')
for d in domains:
    if not settings(d)['require_new_data']:
        print(f'WARNING: {d}: require_new_data = False: staging is swapped even if it is older than the live table (testing only)')

In [ ]:
# ── Folders and Delta facts ─────────────────────────────────────────────────────
def vl_value(vl, key):
    try:
        v = vl[key]
    except Exception:
        return None
    return v.strip().rstrip('/') if isinstance(v, str) and v.strip() else None

def table_path(base, name, s):
    return f"{base}/{s['tables_folder']}/{name}" if s['tables_folder'] else f'{base}/{name}'

def exists(path):
    return notebookutils.fs.exists(path)

def is_delta(path):
    try:
        return exists(path) and DeltaTable.isDeltaTable(spark, path)
    except Exception:
        return False

def rename(src, dst):                         # a folder rename: metadata only, however big the table
    notebookutils.fs.mv(src, dst)

def delete(path):
    notebookutils.fs.rm(path, True)

def read_info(path):
    """Rows (from the Delta statistics), last write, columns and partitioning of a Delta folder."""
    df = spark.read.format('delta').load(path)
    table = DeltaTable.forPath(spark, path)
    return {'rows': df.count(),
            'last_write': table.history(1).collect()[0]['timestamp'],
            'columns': {f.name: f.dataType.simpleString() for f in df.schema.fields},
            'partitions': list(table.detail().collect()[0]['partitionColumns'] or [])}

def minutes_since(moment):
    return round((RUN_AT - moment).total_seconds() / 60, 1)

def seconds_since(start):
    return round(time.time() - start, 1)

def period_key(rng):
    """YYYYPP from the year and period columns (string or number)."""
    return F.col(rng['year_col']).cast('int') * 100 + F.col(rng['period_col']).cast('int')

def range_predicate(rng, start):
    """The same window as SQL, for replaceWhere."""
    return f"CAST(`{rng['year_col']}` AS INT) * 100 + CAST(`{rng['period_col']}` AS INT) >= {start}"

def range_rows(path, rng, start):
    """(rows in the window, rows outside it) of a Delta folder. A row without a year or period is outside."""
    inside = F.coalesce(period_key(rng) >= start, F.lit(False))
    row = spark.read.format('delta').load(path).agg(F.sum(inside.cast('int')).alias('i'), F.count('*').alias('n')).collect()[0]
    return row['i'] or 0, row['n'] - (row['i'] or 0)

def range_facts(live, stg, rng, start):
    """Window counts of live and staging: the costly step, run only once the cheap checks pass."""
    key = period_key(rng)
    row = spark.read.format('delta').load(stg).agg(
        F.sum(F.when(F.coalesce(key >= start, F.lit(False)), 0).otherwise(1)).alias('outside'),
        F.min(key).alias('lo'), F.max(key).alias('hi')).collect()[0]
    live_in, live_out = range_rows(live, rng, start)
    return {'start': start, 'stg_outside': row['outside'] or 0, 'stg_min': row['lo'], 'stg_max': row['hi'],
            'live_in': live_in, 'live_out': live_out}

def latest_version(path):
    return DeltaTable.forPath(spark, path).history(1).collect()[0]['version']

def list_staged(base, s):
    """Tables that have a staging folder next to a live folder."""
    suffix = s['stg_suffix']
    names = [x.name.rstrip('/') for x in notebookutils.fs.ls(f"{base}/{s['tables_folder']}" if s['tables_folder'] else base)]
    return sorted(n[:-len(suffix)] for n in names if n.endswith(suffix) and n[:-len(suffix)] in names)

In [ ]:
# ── The checks: may staging be switched in? ─────────────────────────────────────
def structure_problems(live, other):
    """What differs from live: a missing column, a changed type, other partitioning. New columns are fine."""
    missing = [c for c in live['columns'] if c not in other['columns']]
    retyped = [f"{c} {live['columns'][c]} -> {other['columns'][c]}" for c in live['columns'] if c in other['columns'] and live['columns'][c] != other['columns'][c]]
    parts   = [] if live['partitions'] == other['partitions'] else [f"partitions {live['partitions']} -> {other['partitions']}"]
    return missing + retyped + parts

def first_failed_check(live, stg, checkpoint, s):
    """Decision of the first check that fails, or None when staging may be switched in."""
    if stg is None:                                         return 'NO_SWAP_NO_STAGING'
    if stg['rows'] == 0:                                    return 'NO_SWAP_STG_EMPTY'
    if live and stg['rows'] < live['rows'] * s['min_rows_ratio']: return 'NO_SWAP_STG_SHORT'
    if s['require_new_data'] and checkpoint and stg['last_write'] <= checkpoint: return 'NO_SWAP_NO_NEW_DATA'
    if minutes_since(stg['last_write']) < s['quiet_minutes']: return 'NO_SWAP_STILL_LOADING'
    if live and structure_problems(live, stg):              return 'NO_SWAP_SCHEMA_MISMATCH'
    return None

def first_failed_range_check(live, stg, checkpoint, s, facts):
    """(decision of the first check that fails or None, window counts or None). facts() counts the window."""
    rng = s['range']
    if stg is None:                                         return 'NO_SWAP_NO_STAGING', None
    if stg['rows'] == 0:                                    return 'NO_SWAP_STG_EMPTY', None
    if live is None:                                        return 'NO_SWAP_NO_LIVE', None
    if any(c not in stg['columns'] or c not in live['columns'] for c in (rng['year_col'], rng['period_col'])):
        return 'NO_SWAP_RANGE_COLUMNS', None
    if s['require_new_data'] and checkpoint and stg['last_write'] <= checkpoint: return 'NO_SWAP_NO_NEW_DATA', None
    if minutes_since(stg['last_write']) < s['quiet_minutes']: return 'NO_SWAP_STILL_LOADING', None
    if minutes_since(live['last_write']) < s['quiet_minutes']: return 'NO_SWAP_LIVE_BUSY', None
    if structure_problems(live, stg):                       return 'NO_SWAP_SCHEMA_MISMATCH', None
    f = facts()
    if f['stg_outside']:                                    return 'NO_SWAP_OUT_OF_RANGE', f
    if f['stg_min'] > f['start']:                           return 'NO_SWAP_RANGE_NOT_COVERED', f
    if stg['rows'] < f['live_in'] * s['min_rows_ratio']:    return 'NO_SWAP_STG_SHORT', f
    return None, f

def explain(decision, live, stg, checkpoint, s, rng=None):
    if decision == 'NO_SWAP_NO_NEW_DATA':
        return f"staging last written {stg['last_write']:%Y-%m-%d %H:%M:%S}, not after {checkpoint:%Y-%m-%d %H:%M:%S} (the last swap, or the live table's last write)"
    if decision == 'NO_SWAP_STILL_LOADING':
        return f"staging written {minutes_since(stg['last_write'])} min ago, waits until {s['quiet_minutes']} min"
    if decision == 'NO_SWAP_STG_SHORT' and rng:
        return f"staging has {stg['rows']:,} rows, live has {rng['live_in']:,} in the window from {rng['start']} (needs at least {s['min_rows_ratio']:g} x)"
    if decision == 'NO_SWAP_STG_SHORT':
        return f"staging has {stg['rows']:,} rows, live has {live['rows']:,} (needs at least {s['min_rows_ratio']:g} x)"
    if decision == 'NO_SWAP_LIVE_BUSY':
        return f"live written {minutes_since(live['last_write'])} min ago (a load may be running), waits until {s['quiet_minutes']} min"
    if decision == 'NO_SWAP_RANGE_COLUMNS':
        cols = (s['range']['year_col'], s['range']['period_col'])
        return f"{', '.join(c for c in cols if c not in (stg or {}).get('columns', {}) or c not in (live or {}).get('columns', {}))} missing in staging or live"
    if decision == 'NO_SWAP_OUT_OF_RANGE':
        return f"staging has {rng['stg_outside']:,} rows before period {rng['start']} or without a year / period"
    if decision == 'NO_SWAP_RANGE_NOT_COVERED':
        return f"staging starts at period {rng['stg_min']}, the window at {rng['start']}: live periods in between would be lost"
    if decision == 'NO_SWAP_SCHEMA_MISMATCH':
        return '; '.join(structure_problems(live, stg))
    return None

In [ ]:
# ── The switch: rename, verify, clean up, or roll back ──────────────────────────
def repair_leftovers(live, prev, s):
    """An earlier run died between the renames, or could not delete the previous copy."""
    if not exists(live) and exists(prev):
        if s['dry_run']:
            return 'NEEDS_RECOVERY', 'live table is missing and a previous copy exists: a live run puts it back'
        rename(prev, live)
        return 'RECOVERED', 'live table was missing: the previous copy was put back (checked again next run)'
    if exists(prev) and is_delta(live) and s['delete_prev'] and not s['dry_run']:
        delete(prev)
        return 'CLEANED_PREV', 'a previous copy was left behind and was deleted'
    return None, None

def undo(live, stg, prev, moved_live, moved_stg):
    """Put every folder back where it was."""
    if moved_stg:
        rename(live, stg)
    if moved_live:
        rename(prev, live)

def switch_in_range(r, live, stg, live_info, stg_info, rng, s):
    """Replace the window of live with staging in one Delta commit, verify; on failure restore the version before."""
    v_before, wrote = latest_version(live), False
    r['version_before'] = v_before
    try:
        t = time.time()
        (spark.read.format('delta').load(stg).write.format('delta').mode('overwrite')
              .option('replaceWhere', range_predicate(s['range'], rng['start'])).save(live))
        wrote = True
        r['seconds_write'] = seconds_since(t)

        t = time.time()
        new_in, new_out = range_rows(live, s['range'], rng['start'])
        new_info = read_info(live)
        if new_in != stg_info['rows'] or new_out != rng['live_out'] or structure_problems(live_info, new_info):
            raise RuntimeError(f"verification failed after the write: {new_in:,} rows in the window (staging {stg_info['rows']:,}), "
                               f"{new_out:,} outside (was {rng['live_out']:,})")
        r['seconds_verify'] = seconds_since(t)
    except Exception as e:
        r.update(decision='FAILED', error=str(e)[:400])
        try:
            v_now = latest_version(live)
            if wrote and v_now == v_before + 1:            # our commit is the latest: undo it
                DeltaTable.forPath(spark, live).restoreToVersion(v_before)
                r.update(decision='FAILED_ROLLED_BACK', rolled_back=True)
            elif v_now != v_before:                        # someone else wrote too: restoring would lose their rows
                r['error'] += f' | NOT rolled back: live moved to version {v_now}; restore it to version {v_before} by hand if needed'
        except Exception as e2:
            r['error'] += f' | ROLLBACK FAILED, restore live to version {v_before}: {str(e2)[:200]}'
        return
    r.update(swapped=True, decision='SWAPPED')
    t = time.time()
    try:
        delete(stg)                                        # staging is consumed, as in the full swap
    except Exception as e:
        r.update(decision='SWAPPED_STG_NOT_DELETED', error=f'staging not deleted: {str(e)[:200]}')
    r['seconds_delete'] = seconds_since(t)

def switch_in(r, live, stg, prev, live_info, stg_info, s):
    """live -> prev, staging -> live, verify. On any failure everything is rolled back."""
    moved_live = moved_stg = False
    try:
        t = time.time()
        if live_info:
            rename(live, prev); moved_live = True
        rename(stg, live); moved_stg = True
        r['seconds_rename'] = seconds_since(t)

        t = time.time()
        new_info = read_info(live)
        if new_info['rows'] != stg_info['rows'] or (live_info and structure_problems(live_info, new_info)):
            raise RuntimeError('verification failed after the switch')
        r['seconds_verify'] = seconds_since(t)
    except Exception as e:
        r.update(decision='FAILED', error=str(e)[:400])
        try:
            undo(live, stg, prev, moved_live, moved_stg)
            r.update(decision='FAILED_ROLLED_BACK', rolled_back=moved_live or moved_stg)
        except Exception as e2:
            r['error'] += f' | ROLLBACK FAILED, check {prev}: {str(e2)[:200]}'
        return
    r.update(swapped=True, decision='SWAPPED')
    if moved_live and s['delete_prev']:
        t = time.time()
        try:
            delete(prev)
        except Exception as e:
            r.update(decision='SWAPPED_PREV_NOT_DELETED', error=f'previous copy not deleted: {str(e)[:200]}')
        r['seconds_delete'] = seconds_since(t)

In [ ]:
# ── One table ───────────────────────────────────────────────────────────────────
def new_result(domain, vl_name, name, s):
    """One log row. Same order as LOG_SCHEMA below."""
    return {'run_id': RUN_ID, 'run_at': RUN_AT, 'domain': domain, 'vl_name': vl_name, 'table': name, 'decision': None, 'swapped': False,
            'dry_run': s['dry_run'], 'main_rows': None, 'stg_rows': None, 'main_last_write': None, 'stg_last_write': None, 'checkpoint': None,
            'stg_age_min': None, 'schema_ok': None, 'new_columns': None, 'rolled_back': False, 'seconds_check': None,
            'seconds_rename': None, 'seconds_verify': None, 'seconds_delete': None, 'seconds_total': None, 'error': None,
            'swap_scope': SWAP_SCOPE, 'range_from': None, 'live_rows_in_range': None, 'live_rows_outside': None,
            'stg_min_period': None, 'stg_max_period': None, 'version_before': None, 'seconds_write': None}

def process_range(r, live, stg, last_swap, domain, name, s):
    """The range swap of one table: checks, then replace its window (or WOULD_SWAP in a dry run)."""
    t = time.time()
    stg_info  = read_info(stg)  if is_delta(stg) else None
    live_info = read_info(live) if stg_info and is_delta(live) else None
    checkpoint = max(filter(None, [last_swap.get((domain, name)), live_info and live_info['last_write']]), default=None)
    start = range_start(s['range']['periods'])
    decision, rng = first_failed_range_check(live_info, stg_info, checkpoint, s, lambda: range_facts(live, stg, s['range'], start))
    r['seconds_check'] = seconds_since(t)

    r['range_from'] = str(start)
    if stg_info:
        r.update(stg_rows=stg_info['rows'], stg_last_write=stg_info['last_write'], stg_age_min=minutes_since(stg_info['last_write']), checkpoint=checkpoint)
    if live_info:
        r.update(main_rows=live_info['rows'], main_last_write=live_info['last_write'])
    if live_info and stg_info:
        r.update(schema_ok=not structure_problems(live_info, stg_info), new_columns=', '.join(c for c in stg_info['columns'] if c not in live_info['columns']) or None)
    if rng:
        r.update(live_rows_in_range=rng['live_in'], live_rows_outside=rng['live_out'],
                 stg_min_period=None if rng['stg_min'] is None else str(rng['stg_min']),
                 stg_max_period=None if rng['stg_max'] is None else str(rng['stg_max']))

    if decision:
        r.update(decision=decision, error=explain(decision, live_info, stg_info, checkpoint, s, rng))
    elif s['dry_run']:
        r['decision'] = 'WOULD_SWAP'
    else:
        switch_in_range(r, live, stg, live_info, stg_info, rng, s)

def process_table(domain, vl_name, base, name, last_swap, s):
    r, started = new_result(domain, vl_name, name, s), time.time()
    live = table_path(base, name, s)
    stg  = table_path(base, name + s['stg_suffix'], s)
    prev = table_path(base, name + s['prev_suffix'], s)
    if SWAP_SCOPE == 'range' and not s.get('range'):
        r.update(decision='SKIPPED_NOT_RANGE', seconds_total=seconds_since(started))
        return r
    try:
        r['decision'], r['error'] = repair_leftovers(live, prev, s)
        if r['decision']:
            return r
        if SWAP_SCOPE == 'range':
            process_range(r, live, stg, last_swap, domain, name, s)
            r['seconds_total'] = seconds_since(started)
            return r

        # read the facts: staging first, so a table with nothing staged costs almost nothing
        t = time.time()
        stg_info  = read_info(stg)  if is_delta(stg) else None
        live_info = read_info(live) if stg_info and is_delta(live) else None
        # new data = written after the last swap (or after the live table's last write)
        checkpoint = max(filter(None, [last_swap.get((domain, name)), live_info and live_info['last_write']]), default=None)
        decision = first_failed_check(live_info, stg_info, checkpoint, s)
        r['seconds_check'] = seconds_since(t)

        if stg_info:
            r.update(stg_rows=stg_info['rows'], stg_last_write=stg_info['last_write'], stg_age_min=minutes_since(stg_info['last_write']), checkpoint=checkpoint)
        if live_info:
            r.update(main_rows=live_info['rows'], main_last_write=live_info['last_write'])
        if live_info and stg_info:
            r.update(schema_ok=not structure_problems(live_info, stg_info), new_columns=', '.join(c for c in stg_info['columns'] if c not in live_info['columns']) or None)

        if decision:
            r.update(decision=decision, error=explain(decision, live_info, stg_info, checkpoint, s))
        elif s['dry_run']:
            r['decision'] = 'WOULD_SWAP'
        else:
            switch_in(r, live, stg, prev, live_info, stg_info, s)
    except Exception as e:
        r.update(decision='FAILED', error=str(e)[:400])
    r['seconds_total'] = seconds_since(started)
    return r

In [ ]:
# ── The log (b1_swap_log) and the check mark ─────────────────────────────────────
LOG_SCHEMA = ('run_id string, run_at timestamp, domain string, vl_name string, `table` string, decision string, swapped boolean, dry_run boolean, '
              'main_rows long, stg_rows long, main_last_write timestamp, stg_last_write timestamp, checkpoint timestamp, stg_age_min double, '
              'schema_ok boolean, new_columns string, rolled_back boolean, seconds_check double, seconds_rename double, seconds_verify double, '
              'seconds_delete double, seconds_total double, error string, '
              'swap_scope string, range_from string, live_rows_in_range long, live_rows_outside long, stg_min_period string, '
              'stg_max_period string, version_before long, seconds_write double')

def log_path(cfg, vl, base):
    """LOG_TABLE_PATH, else the domain's log_abfss, else OPS_LH_ABFSS of its Variable Library, else its own lakehouse."""
    if LOG_TABLE_PATH:
        return LOG_TABLE_PATH.rstrip('/')
    ops = (cfg.get('log_abfss') or '').strip().rstrip('/') or (vl_value(vl, 'OPS_LH_ABFSS') if vl else None) or base
    schema = (vl_value(vl, 'OPS_SCHEMA') if vl else None) or 'dbo'
    return f'{ops}/{schema}/b1_swap_log' if ops else None

def read_last_swaps(path, domain):
    """{(domain, table): when it was last swapped}: the check mark for the next run."""
    if not path or not DeltaTable.isDeltaTable(spark, path):
        return {}
    rows = (spark.read.format('delta').load(path).filter((F.col('domain') == domain) & F.col('swapped') & ~F.col('dry_run'))
            .groupBy('domain', 'table').agg(F.max('run_at').alias('at')).collect())
    return {(r['domain'], r['table']): r['at'] for r in rows}

def write_log(path, results):
    df = spark.createDataFrame([tuple(r.values()) for r in results], LOG_SCHEMA)
    for attempt in range(4):                   # retry when several writers append at once
        try:
            df.write.format('delta').mode('append').option('mergeSchema', 'true').save(path)
            return
        except Exception as e:
            if attempt == 3:
                print(f'  WARNING: log write failed ({str(e)[:150]})')
            time.sleep(2 ** attempt)

In [ ]:
# ── Run the domains side by side, the tables of a domain in order ───────────────
def failed_row(domain, cfg, name, s, error):
    return dict(new_result(domain, cfg.get('vl') or '(address)', name, s), decision='FAILED', error=error)

def lakehouse_of(cfg):
    """(address of the Tables folder, Variable Library or None)."""
    if cfg.get('abfss'):
        return cfg['abfss'].strip().rstrip('/'), None
    vl = notebookutils.variableLibrary.getLibrary(cfg['vl'])
    return vl_value(vl, cfg['lh_key']), vl

def run_domain(domain):
    """(log rows, log path) of one domain. Nothing here raises: a problem becomes a FAILED row."""
    cfg, s0 = SWAP_DOMAINS[domain], settings(domain)
    try:
        base, vl = lakehouse_of(cfg)
    except Exception as e:
        return [failed_row(domain, cfg, '(all)', s0, f'lakehouse address not found: {str(e)[:200]}')], None
    if not base:
        return [failed_row(domain, cfg, '(all)', s0, f"VL key {cfg.get('lh_key')} not found in {cfg.get('vl')}")], None
    path = log_path(cfg, vl, base)
    try:
        names = list_staged(base, s0) if cfg['tables'] == ['*'] else list(cfg['tables'])
    except Exception as e:
        return [failed_row(domain, cfg, '(all)', s0, f'cannot list {base}: {str(e)[:200]}')], path
    names = [n for n in names if not only or n in only or f'{domain}.{n}' in only]
    if names and cfg['tables'] != ['*'] and not any(exists(table_path(base, n + suffix, s0)) for n in names for suffix in ('', s0['stg_suffix'], s0['prev_suffix'])):
        # a wrong address or folder would otherwise look like "no staging" for every table (live, staging or previous folder found = right place)
        return [failed_row(domain, cfg, '(all)', s0, f"none of the {len(names)} tables exist under {table_path(base, '<table>', s0)}: check the address and tables_folder")], path
    last_swap = read_last_swaps(path, domain)

    def one(name):
        return process_table(domain, cfg.get('vl') or '(address)', base, name, last_swap, settings(domain, name))

    if s0['parallel_tables'] > 1 and len(names) > 1:
        with ThreadPoolExecutor(max_workers=s0['parallel_tables']) as pool:
            return list(pool.map(one, names)), path
    return [one(n) for n in names], path

with ThreadPoolExecutor(max_workers=max(1, min(MAX_PARALLEL_DOMAINS, len(domains)))) as pool:
    outcomes = list(pool.map(run_domain, domains))

results = []
for domain, (domain_results, path) in zip(domains, outcomes):
    for r in domain_results:
        print(f"  {domain:<10} {r['table']:<12} {r['decision']:<26} live {r['main_rows'] or '-':>11}  staging {r['stg_rows'] or '-':>11}  {r['seconds_total'] or 0:>5}s" + (f"  {r['error']}" if r['error'] else ''))
    if path and domain_results:
        write_log(path, domain_results)
    elif domain_results:
        print(f'  ({domain}: no log location, results not saved)')
    results += domain_results

In [ ]:
# ── Summary in plain language ───────────────────────────────────────────────────
MEANING = {
    'SWAPPED': 'swapped: staging is now the live table',
    'WOULD_SWAP': 'would swap (dry run, nothing changed)',
    'SWAPPED_PREV_NOT_DELETED': 'swapped, but the previous copy could not be deleted (retried next run)',
    'SWAPPED_STG_NOT_DELETED': 'window replaced, but staging could not be deleted (delete it before the next load)',
    'SKIPPED_NOT_RANGE': 'range scope: table has no range settings, not swapped',
    'NO_SWAP_NO_LIVE': 'no live table: a range swap only replaces part of one',
    'NO_SWAP_RANGE_COLUMNS': 'year or period column missing: not swapped',
    'NO_SWAP_LIVE_BUSY': 'live written a moment ago: a load may still be running',
    'NO_SWAP_OUT_OF_RANGE': 'staging has rows before the window: not swapped',
    'NO_SWAP_RANGE_NOT_COVERED': 'staging starts after the window start: not swapped',
    'NO_SWAP_NO_STAGING': 'no staging table: nothing was loaded',
    'NO_SWAP_NO_NEW_DATA': 'staging not written since the last swap',
    'NO_SWAP_STILL_LOADING': 'staging written a moment ago: the load may still be running',
    'NO_SWAP_STG_EMPTY': 'staging is empty: not swapped',
    'NO_SWAP_STG_SHORT': 'staging has fewer rows than live: not swapped',
    'NO_SWAP_SCHEMA_MISMATCH': 'staging structure differs from live: not swapped',
    'RECOVERED': 'live table was missing: previous copy put back',
    'NEEDS_RECOVERY': 'live table is missing, previous copy exists (a live run puts it back)',
    'CLEANED_PREV': 'left-over previous copy deleted',
    'FAILED_ROLLED_BACK': 'FAILED after the switch: rolled back, live table unchanged',
    'FAILED': 'FAILED',
}
NEEDS_ATTENTION = ('NEEDS_RECOVERY', 'NO_SWAP_STG_EMPTY', 'NO_SWAP_STG_SHORT', 'NO_SWAP_SCHEMA_MISMATCH', 'SWAPPED_PREV_NOT_DELETED',
                   'SWAPPED_STG_NOT_DELETED', 'NO_SWAP_NO_LIVE', 'NO_SWAP_RANGE_COLUMNS', 'NO_SWAP_OUT_OF_RANGE', 'NO_SWAP_RANGE_NOT_COVERED')

dry = sum(1 for r in results if r['dry_run'])
mode = 'DRY RUN' if dry == len(results) else 'LIVE' if dry == 0 else f'MIXED ({dry} dry, {len(results) - dry} live)'
print(f'B1 swap summary · {mode} · scope {SWAP_SCOPE} · {len(domains)} domain(s) · {len(results)} tables')
print(f"\n  {'domain':<12}{'tables':>7}{'swapped':>9}{'would':>7}{'failed':>8}{'attention':>11}")
for d in domains:
    rs = [r for r in results if r['domain'] == d]
    print(f"  {d:<12}{len(rs):>7}{sum(r['swapped'] for r in rs):>9}{sum(r['decision'] == 'WOULD_SWAP' for r in rs):>7}"
          f"{sum(r['decision'].startswith('FAILED') for r in rs):>8}{sum(r['decision'] in NEEDS_ATTENTION for r in rs):>11}")
print()
for decision in sorted({r['decision'] for r in results}):
    print(f"  {sum(r['decision'] == decision for r in results):>4}  {MEANING.get(decision, decision)}")

attention = [r for r in results if r['decision'].startswith('FAILED') or r['decision'] in NEEDS_ATTENTION]
if attention:
    print('\nNeeds attention:')
    for r in attention:
        print(f"  {r['domain']}.{r['table']:<12} {MEANING.get(r['decision'], r['decision'])}" + (f": {r['error']}" if r['error'] else ''))

swapped = [r for r in results if r['swapped']]
if swapped:
    print(f"\nSwapped:\n  {'domain.table':<24}{'rows':>14}{'switch s':>10}{'verify s':>10}{'total s':>9}  window")
    for r in swapped:
        print(f"  {r['domain'] + '.' + r['table']:<24}{r['stg_rows']:>14,}{r['seconds_rename'] or r['seconds_write'] or 0:>10}"
              f"{r['seconds_verify'] or 0:>10}{r['seconds_total'] or 0:>9}  " + (f"from {r['range_from']}" if r['range_from'] else 'whole table'))

if any(r['decision'].startswith('FAILED') for r in results):
    raise Exception('B1 swap: at least one table failed, see the summary above')

In [ ]:
# ── End of run: stop the session to release capacity (15 Nov 2026) ──────────────────────────────────────
# Same rules as end_session() in NB_Utils_Common, which this notebook does not load: skipped in a pipeline run
# (Fabric stops the session itself) and in a reference run; set END_SESSION = False in a cell above to keep it.
if globals().get('END_SESSION', True):
    try:
        _ctx = notebookutils.runtime.context
        if not (_ctx.get('isForPipeline') or _ctx.get('isReferenceRun')):
            print('Session stopped to release capacity (END_SESSION = False keeps it)')
            notebookutils.session.stop()
    except Exception as e:
        if 'STOP_SESSION_REQUEST' in str(e):
            raise    # Fabric stops the session through this marker exception: it must reach the runtime (18 Nov 2026)
        print(f'Session not stopped ({str(e)[:120]})')